## 1. Imports and Project Paths

In [2]:
from pathlib import Path

import pandas as pd


ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"
REPORTS_DIR = ROOT_DIR / "reports"

LEADS_FILE = RAW_DATA_DIR / "leads.csv"
EVENTS_FILE = RAW_DATA_DIR / "lead_stage_events.csv"

CLEAN_LEADS_FILE = PROCESSED_DATA_DIR / "clean_leads.csv"
CLEAN_EVENTS_FILE = PROCESSED_DATA_DIR / "clean_funnel_events.csv"
QUALITY_REPORT_FILE = REPORTS_DIR / "data_quality_report.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")


Project root: c:\Projects\b2b-growth-funnel-analytics


## 2. Load Raw Data

In [3]:
leads_raw = pd.read_csv(LEADS_FILE)
events_raw = pd.read_csv(EVENTS_FILE)

print(f"Leads:  {leads_raw.shape}")
print(f"Events: {events_raw.shape}")


Leads:  (500, 8)
Events: (1445, 4)


In [4]:
leads_raw.head()

,lead_id,created_date,channel,campaign,region,company_size,industry,acquisition_cost
0,L00001,2025-03-25,Webinar,Finance Automation Webinar,US,Enterprise,Healthcare,38.15
1,L00002,2025-06-11,Email,Product Update,US,Enterprise,Healthcare,26.38
2,L00003,2025-04-10,Organic Search,SEO Evergreen,US,SMB,Retail,12.34
3,L00004,2025-06-02,Organic Search,Blog CTA,US,SMB,Logistics,11.88
4,L00005,2025-04-29,Referral,Customer Referral,UK,Mid-Market,Healthcare,9.08


In [5]:
events_raw.head()

,lead_id,stage,stage_date,revenue
0,L00001,Lead,2025-03-25,0.0
1,L00001,Lost,2025-04-04,0.0
2,L00002,Lead,2025-06-11,0.0
3,L00002,MQL,2025-06-24,0.0
4,L00002,Lost,2025-07-13,0.0


## 3. Validation Rules

These rules define what the project considers valid source data.

We intentionally keep business logic here rather than silently correcting serious issues.  
If a critical validation fails, the notebook will still generate the quality report but will not export processed data.


In [6]:
EXPECTED_LEAD_COLUMNS = [
    "lead_id",
    "created_date",
    "channel",
    "campaign",
    "region",
    "company_size",
    "industry",
    "acquisition_cost",
]

EXPECTED_EVENT_COLUMNS = [
    "lead_id",
    "stage",
    "stage_date",
    "revenue",
]

TEXT_LEAD_COLUMNS = [
    "lead_id",
    "channel",
    "campaign",
    "region",
    "company_size",
    "industry",
]

ALLOWED_STAGES = {"Lead", "MQL", "SQL", "Customer", "Lost"}

SUCCESS_STAGE_ORDER = {
    "Lead": 1,
    "MQL": 2,
    "SQL": 3,
    "Customer": 4,
}


## 4. Data Quality Report Helper

In [7]:
quality_results = []


def add_check(
    category: str,
    check_name: str,
    issues: int,
    severity: str,
    description: str,
) -> None:
    quality_results.append(
        {
            "category": category,
            "check_name": check_name,
            "severity": severity,
            "issues": int(issues),
            "status": "PASS" if issues == 0 else "FAIL",
            "description": description,
        }
    )


## 5. Schema Validation

In [8]:
missing_lead_columns = [
    column for column in EXPECTED_LEAD_COLUMNS
    if column not in leads_raw.columns
]

missing_event_columns = [
    column for column in EXPECTED_EVENT_COLUMNS
    if column not in events_raw.columns
]

unexpected_lead_columns = [
    column for column in leads_raw.columns
    if column not in EXPECTED_LEAD_COLUMNS
]

unexpected_event_columns = [
    column for column in events_raw.columns
    if column not in EXPECTED_EVENT_COLUMNS
]

add_check(
    "Schema",
    "Missing required lead columns",
    len(missing_lead_columns),
    "CRITICAL",
    "All required columns must exist in leads.csv.",
)

add_check(
    "Schema",
    "Missing required event columns",
    len(missing_event_columns),
    "CRITICAL",
    "All required columns must exist in lead_stage_events.csv.",
)

add_check(
    "Schema",
    "Unexpected lead columns",
    len(unexpected_lead_columns),
    "WARNING",
    "Unexpected columns should be reviewed before ingestion.",
)

add_check(
    "Schema",
    "Unexpected event columns",
    len(unexpected_event_columns),
    "WARNING",
    "Unexpected columns should be reviewed before ingestion.",
)

print("Missing lead columns:", missing_lead_columns)
print("Missing event columns:", missing_event_columns)
print("Unexpected lead columns:", unexpected_lead_columns)
print("Unexpected event columns:", unexpected_event_columns)


Missing lead columns: []
Missing event columns: []
Unexpected lead columns: []
Unexpected event columns: []


## 6. Create Working Copies and Standardize Basic Types

In [9]:
leads = leads_raw.copy()
events = events_raw.copy()

# Convert empty/whitespace-only strings to missing values.
leads = leads.replace(r"^\s*$", pd.NA, regex=True)
events = events.replace(r"^\s*$", pd.NA, regex=True)

# Trim text fields without changing business capitalization.
for column in TEXT_LEAD_COLUMNS:
    leads[column] = leads[column].astype("string").str.strip()

events["lead_id"] = events["lead_id"].astype("string").str.strip()
events["stage"] = events["stage"].astype("string").str.strip()

# Parse dates.
leads["created_date"] = pd.to_datetime(
    leads["created_date"],
    errors="coerce",
)

events["stage_date"] = pd.to_datetime(
    events["stage_date"],
    errors="coerce",
)

# Parse numeric fields.
leads["acquisition_cost"] = pd.to_numeric(
    leads["acquisition_cost"],
    errors="coerce",
)

events["revenue"] = pd.to_numeric(
    events["revenue"],
    errors="coerce",
)


## 7. Missing Values and Duplicate Validation

In [10]:
required_lead_nulls = leads[EXPECTED_LEAD_COLUMNS].isna().sum().sum()
required_event_nulls = events[EXPECTED_EVENT_COLUMNS].isna().sum().sum()

duplicate_lead_rows = leads.duplicated().sum()
duplicate_event_rows = events.duplicated().sum()
duplicate_lead_ids = leads["lead_id"].duplicated().sum()

add_check(
    "Completeness",
    "Missing values in lead fields",
    required_lead_nulls,
    "CRITICAL",
    "Required lead fields should not contain missing values.",
)

add_check(
    "Completeness",
    "Missing values in event fields",
    required_event_nulls,
    "CRITICAL",
    "Required event fields should not contain missing values.",
)

add_check(
    "Duplicates",
    "Duplicate lead rows",
    duplicate_lead_rows,
    "WARNING",
    "Exact duplicate rows should be investigated.",
)

add_check(
    "Duplicates",
    "Duplicate event rows",
    duplicate_event_rows,
    "WARNING",
    "Exact duplicate event rows should be investigated.",
)

add_check(
    "Keys",
    "Duplicate lead IDs",
    duplicate_lead_ids,
    "CRITICAL",
    "lead_id must uniquely identify a lead.",
)

print(f"Missing lead values: {required_lead_nulls:,}")
print(f"Missing event values: {required_event_nulls:,}")
print(f"Duplicate lead rows: {duplicate_lead_rows:,}")
print(f"Duplicate event rows: {duplicate_event_rows:,}")
print(f"Duplicate lead IDs: {duplicate_lead_ids:,}")


Missing lead values: 0
Missing event values: 0
Duplicate lead rows: 0
Duplicate event rows: 0
Duplicate lead IDs: 0


## 8. Referential Integrity

In [11]:
orphan_events = events.loc[
    ~events["lead_id"].isin(leads["lead_id"])
]

leads_without_events = leads.loc[
    ~leads["lead_id"].isin(events["lead_id"])
]

add_check(
    "Relationships",
    "Orphan funnel events",
    len(orphan_events),
    "CRITICAL",
    "Every event must belong to an existing lead.",
)

add_check(
    "Relationships",
    "Leads without funnel events",
    len(leads_without_events),
    "CRITICAL",
    "Every lead should have at least one funnel event.",
)

print(f"Orphan events: {len(orphan_events):,}")
print(f"Leads without events: {len(leads_without_events):,}")


Orphan events: 0
Leads without events: 0


## 9. Categorical and Funnel-Stage Validation

In [12]:
unknown_stages = events.loc[
    ~events["stage"].isin(ALLOWED_STAGES)
]

add_check(
    "Business Rules",
    "Unknown funnel stages",
    len(unknown_stages),
    "CRITICAL",
    f"Allowed stages are: {sorted(ALLOWED_STAGES)}.",
)

# Every lead should have one initial Lead event.
lead_stage_counts = (
    events.loc[events["stage"] == "Lead"]
    .groupby("lead_id")
    .size()
)

missing_initial_lead_stage = (
    ~leads["lead_id"].isin(lead_stage_counts.index)
).sum()

multiple_initial_lead_stages = (
    lead_stage_counts > 1
).sum()

add_check(
    "Business Rules",
    "Leads missing initial Lead stage",
    missing_initial_lead_stage,
    "CRITICAL",
    "Every lead should begin with a Lead event.",
)

add_check(
    "Business Rules",
    "Multiple initial Lead events",
    multiple_initial_lead_stages,
    "CRITICAL",
    "Each lead should have only one initial Lead event.",
)

print(f"Unknown stages: {len(unknown_stages):,}")
print(f"Leads missing initial Lead stage: {missing_initial_lead_stage:,}")
print(f"Leads with multiple Lead events: {multiple_initial_lead_stages:,}")


Unknown stages: 0
Leads missing initial Lead stage: 0
Leads with multiple Lead events: 0


## 10. Numeric Validation

In [13]:
negative_acquisition_cost = (
    leads["acquisition_cost"] < 0
).sum()

negative_revenue = (
    events["revenue"] < 0
).sum()

add_check(
    "Numeric",
    "Negative acquisition cost",
    negative_acquisition_cost,
    "CRITICAL",
    "Acquisition cost cannot be negative.",
)

add_check(
    "Numeric",
    "Negative revenue",
    negative_revenue,
    "CRITICAL",
    "Revenue cannot be negative.",
)

print(f"Negative acquisition costs: {negative_acquisition_cost:,}")
print(f"Negative revenue values: {negative_revenue:,}")


Negative acquisition costs: 0
Negative revenue values: 0


## 11. Date Validation

In [14]:
invalid_lead_dates = leads["created_date"].isna().sum()
invalid_event_dates = events["stage_date"].isna().sum()

date_validation = events.merge(
    leads[["lead_id", "created_date"]],
    on="lead_id",
    how="left",
)

events_before_creation = date_validation.loc[
    date_validation["stage_date"]
    < date_validation["created_date"]
]

add_check(
    "Dates",
    "Invalid lead creation dates",
    invalid_lead_dates,
    "CRITICAL",
    "created_date must contain valid dates.",
)

add_check(
    "Dates",
    "Invalid funnel event dates",
    invalid_event_dates,
    "CRITICAL",
    "stage_date must contain valid dates.",
)

add_check(
    "Dates",
    "Events before lead creation",
    len(events_before_creation),
    "CRITICAL",
    "A funnel event cannot occur before lead creation.",
)

print(f"Invalid lead dates: {invalid_lead_dates:,}")
print(f"Invalid event dates: {invalid_event_dates:,}")
print(f"Events before lead creation: {len(events_before_creation):,}")


Invalid lead dates: 0
Invalid event dates: 0
Events before lead creation: 0


## 12. Revenue Consistency

In [15]:
customer_events = events.loc[
    events["stage"] == "Customer"
]

non_customer_events = events.loc[
    events["stage"] != "Customer"
]

customers_without_positive_revenue = (
    customer_events["revenue"].fillna(0) <= 0
).sum()

non_customers_with_revenue = (
    non_customer_events["revenue"].fillna(0) > 0
).sum()

add_check(
    "Revenue",
    "Customers without positive revenue",
    customers_without_positive_revenue,
    "CRITICAL",
    "Customer conversion events should contain positive revenue.",
)

add_check(
    "Revenue",
    "Non-customer events with revenue",
    non_customers_with_revenue,
    "CRITICAL",
    "Revenue should only be recorded on Customer events.",
)

print(
    "Customers without positive revenue:",
    f"{customers_without_positive_revenue:,}",
)
print(
    "Non-customer events with revenue:",
    f"{non_customers_with_revenue:,}",
)


Customers without positive revenue: 0
Non-customer events with revenue: 0


## 13. Funnel Prerequisite Validation

In [16]:
stage_presence = (
    events.assign(present=1)
    .pivot_table(
        index="lead_id",
        columns="stage",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
)

for stage in ALLOWED_STAGES:
    if stage not in stage_presence.columns:
        stage_presence[stage] = 0

missing_prerequisites = (
    ((stage_presence["MQL"] == 1) & (stage_presence["Lead"] == 0))
    | (
        (stage_presence["SQL"] == 1)
        & (
            (stage_presence["Lead"] == 0)
            | (stage_presence["MQL"] == 0)
        )
    )
    | (
        (stage_presence["Customer"] == 1)
        & (
            (stage_presence["Lead"] == 0)
            | (stage_presence["MQL"] == 0)
            | (stage_presence["SQL"] == 0)
        )
    )
).sum()

add_check(
    "Funnel Logic",
    "Missing prerequisite funnel stages",
    missing_prerequisites,
    "CRITICAL",
    "MQL requires Lead, SQL requires Lead + MQL, and Customer requires Lead + MQL + SQL.",
)

print(f"Leads with missing prerequisite stages: {missing_prerequisites:,}")


Leads with missing prerequisite stages: 0


## 14. Funnel Sequence Validation

In [17]:
sequence_events = events.loc[
    events["stage"].isin(SUCCESS_STAGE_ORDER)
].copy()

sequence_events["stage_rank"] = (
    sequence_events["stage"]
    .map(SUCCESS_STAGE_ORDER)
)

invalid_sequence_leads = []

for lead_id, group in sequence_events.groupby("lead_id"):
    ordered = group.sort_values(
        ["stage_date", "stage_rank"]
    )
    ranks = ordered["stage_rank"].tolist()

    if any(
        current < previous
        for previous, current in zip(ranks, ranks[1:])
    ):
        invalid_sequence_leads.append(lead_id)

add_check(
    "Funnel Logic",
    "Out-of-order funnel stages",
    len(invalid_sequence_leads),
    "CRITICAL",
    "Successful stages must progress Lead → MQL → SQL → Customer.",
)

print(
    "Leads with out-of-order successful stages:",
    f"{len(invalid_sequence_leads):,}",
)


Leads with out-of-order successful stages: 0


## 15. Duplicate Funnel Stage Validation

In [18]:
duplicate_stage_events = (
    events.groupby(["lead_id", "stage"])
    .size()
    .gt(1)
    .sum()
)

add_check(
    "Funnel Logic",
    "Duplicate stage events per lead",
    duplicate_stage_events,
    "CRITICAL",
    "A lead should reach each funnel stage at most once in this dataset.",
)

print(
    "Lead-stage combinations occurring more than once:",
    f"{duplicate_stage_events:,}",
)


Lead-stage combinations occurring more than once: 0


## 16. Terminal Outcome Validation

In [19]:
terminal_events = events.loc[
    events["stage"].isin(["Customer", "Lost"])
]

terminal_event_counts = (
    terminal_events.groupby("lead_id")
    .size()
)

leads_without_terminal_outcome = (
    ~leads["lead_id"].isin(terminal_event_counts.index)
).sum()

multiple_terminal_events = (
    terminal_event_counts > 1
).sum()

both_customer_and_lost = (
    terminal_events.groupby("lead_id")["stage"]
    .nunique()
    .gt(1)
    .sum()
)

add_check(
    "Funnel Logic",
    "Leads without terminal outcome",
    leads_without_terminal_outcome,
    "CRITICAL",
    "Each lead should end as either Customer or Lost.",
)

add_check(
    "Funnel Logic",
    "Multiple terminal events",
    multiple_terminal_events,
    "CRITICAL",
    "A lead should have one terminal event.",
)

add_check(
    "Funnel Logic",
    "Leads marked both Customer and Lost",
    both_customer_and_lost,
    "CRITICAL",
    "Customer and Lost are mutually exclusive outcomes.",
)

print(
    "Leads without terminal outcome:",
    f"{leads_without_terminal_outcome:,}",
)
print(
    "Leads with multiple terminal events:",
    f"{multiple_terminal_events:,}",
)
print(
    "Leads marked both Customer and Lost:",
    f"{both_customer_and_lost:,}",
)


Leads without terminal outcome: 0
Leads with multiple terminal events: 0
Leads marked both Customer and Lost: 0


## 17. Clean and Standardize the Data

At this point, serious data problems have already been identified.

The transformations below are intentionally conservative:
- remove exact duplicate rows only
- standardize text whitespace
- use proper date/numeric types
- round currency fields
- sort records consistently
- add a stable `event_id`

We do **not** invent missing business values or repair invalid funnel logic automatically.


In [20]:
clean_leads = leads.copy()
clean_events = events.copy()

# Exact duplicate rows are safe to remove.
clean_leads = clean_leads.drop_duplicates().copy()
clean_events = clean_events.drop_duplicates().copy()

# Round monetary fields.
clean_leads["acquisition_cost"] = (
    clean_leads["acquisition_cost"].round(2)
)

clean_events["revenue"] = (
    clean_events["revenue"].round(2)
)

# Sort leads consistently.
clean_leads = (
    clean_leads
    .sort_values(["created_date", "lead_id"])
    .reset_index(drop=True)
)

# Sort events chronologically within each lead.
clean_events["_stage_rank"] = (
    clean_events["stage"]
    .map({
        "Lead": 1,
        "MQL": 2,
        "SQL": 3,
        "Customer": 4,
        "Lost": 5,
    })
    .fillna(99)
)

clean_events = (
    clean_events
    .sort_values(
        ["lead_id", "stage_date", "_stage_rank"]
    )
    .drop(columns="_stage_rank")
    .reset_index(drop=True)
)

# Add a stable event key for the relational database.
clean_events.insert(
    0,
    "event_id",
    [
        f"EVT{i:06d}"
        for i in range(1, len(clean_events) + 1)
    ],
)

print(clean_leads.shape)
print(clean_events.shape)


(500, 8)
(1445, 5)


## 18. Review Cleaned Data

In [21]:
clean_leads.head()

,lead_id,created_date,channel,campaign,region,company_size,industry,acquisition_cost
0,L00052,2025-01-01,Referral,Partner Referral,Canada,Mid-Market,SaaS,10.89
1,L00109,2025-01-01,Google Ads,Search Campaign,US,SMB,Logistics,66.52
2,L00197,2025-01-01,Webinar,Finance Automation Webinar,Canada,Mid-Market,Finance,65.79
3,L00327,2025-01-01,Google Ads,Search Campaign,Canada,SMB,SaaS,96.07
4,L00413,2025-01-02,Email,Product Update,US,Mid-Market,SaaS,13.52


In [22]:
clean_events.head(10)

,event_id,lead_id,stage,stage_date,revenue
0,EVT000001,L00001,Lead,2025-03-25,0.00
1,EVT000002,L00001,Lost,2025-04-04,0.00
2,EVT000003,L00002,Lead,2025-06-11,0.00
3,EVT000004,L00002,MQL,2025-06-24,0.00
4,EVT000005,L00002,Lost,2025-07-13,0.00
5,EVT000006,L00003,Lead,2025-04-10,0.00
6,EVT000007,L00003,MQL,2025-04-14,0.00
7,EVT000008,L00003,SQL,2025-04-18,0.00
8,EVT000009,L00003,Customer,2025-05-05,6804.92
9,EVT000010,L00004,Lead,2025-06-02,0.00


## 19. Build the Final Data Quality Report

In [23]:
quality_report = pd.DataFrame(quality_results)

severity_order = {
    "CRITICAL": 1,
    "WARNING": 2,
}

quality_report["_severity_order"] = (
    quality_report["severity"]
    .map(severity_order)
)

quality_report = (
    quality_report
    .sort_values(
        ["_severity_order", "category", "check_name"]
    )
    .drop(columns="_severity_order")
    .reset_index(drop=True)
)

quality_report


,category,check_name,severity,issues,status,description
0,Business Rules,Leads missing initial Lead stage,CRITICAL,0,PASS,Every lead should begin with a Lead event.
1,Business Rules,Multiple initial Lead events,CRITICAL,0,PASS,Each lead should have only one initial Lead ev...
2,Business Rules,Unknown funnel stages,CRITICAL,0,PASS,"Allowed stages are: ['Customer', 'Lead', 'Lost..."
3,Completeness,Missing values in event fields,CRITICAL,0,PASS,Required event fields should not contain missi...
4,Completeness,Missing values in lead fields,CRITICAL,0,PASS,Required lead fields should not contain missin...
5,Dates,Events before lead creation,CRITICAL,0,PASS,A funnel event cannot occur before lead creation.
6,Dates,Invalid funnel event dates,CRITICAL,0,PASS,stage_date must contain valid dates.
7,Dates,Invalid lead creation dates,CRITICAL,0,PASS,created_date must contain valid dates.
8,Funnel Logic,Duplicate stage events per lead,CRITICAL,0,PASS,A lead should reach each funnel stage at most ...
9,Funnel Logic,Leads marked both Customer and Lost,CRITICAL,0,PASS,Customer and Lost are mutually exclusive outco...


In [24]:
summary = (
    quality_report.groupby(["severity", "status"])
    .size()
    .rename("checks")
    .reset_index()
)

summary


,severity,status,checks
0,CRITICAL,PASS,23
1,WARNING,PASS,4


## 20. Export Quality Report and Processed Data

In [25]:
# Always export the quality report.
quality_report.to_csv(
    QUALITY_REPORT_FILE,
    index=False,
)

critical_failures = quality_report.loc[
    (quality_report["severity"] == "CRITICAL")
    & (quality_report["status"] == "FAIL")
]

if not critical_failures.empty:
    print("Processed files were NOT exported.")
    print(
        f"{len(critical_failures)} critical validation "
        "check(s) failed."
    )
    display(
        critical_failures[
            [
                "category",
                "check_name",
                "issues",
                "description",
            ]
        ]
    )
else:
    clean_leads.to_csv(
        CLEAN_LEADS_FILE,
        index=False,
        date_format="%Y-%m-%d",
    )

    clean_events.to_csv(
        CLEAN_EVENTS_FILE,
        index=False,
        date_format="%Y-%m-%d",
    )

    print("Validation passed.")
    print(f"Saved: {CLEAN_LEADS_FILE}")
    print(f"Saved: {CLEAN_EVENTS_FILE}")
    print(f"Saved: {QUALITY_REPORT_FILE}")


Validation passed.
Saved: c:\Projects\b2b-growth-funnel-analytics\data\processed\clean_leads.csv
Saved: c:\Projects\b2b-growth-funnel-analytics\data\processed\clean_funnel_events.csv
Saved: c:\Projects\b2b-growth-funnel-analytics\reports\data_quality_report.csv


## 21. Final Verification

In [26]:
if CLEAN_LEADS_FILE.exists() and CLEAN_EVENTS_FILE.exists():
    exported_leads = pd.read_csv(CLEAN_LEADS_FILE)
    exported_events = pd.read_csv(CLEAN_EVENTS_FILE)

    verification = pd.DataFrame({
        "dataset": [
            "clean_leads",
            "clean_funnel_events",
        ],
        "rows": [
            len(exported_leads),
            len(exported_events),
        ],
        "columns": [
            exported_leads.shape[1],
            exported_events.shape[1],
        ],
    })

    display(verification)
else:
    print(
        "Processed datasets are unavailable because "
        "critical validation checks failed."
    )


,dataset,rows,columns
0,clean_leads,500,8
1,clean_funnel_events,1445,5
